# Cross-Sectional Signal Validation in Indian Equities
### A Research Note on Signal Efficacy, Factor Premia, Portfolio Construction, and Transaction Costs

**Universe:** 20 NSE large-caps (Nifty constituents) &nbsp;|&nbsp; **Period:** 2022-01-01 → 2026-01-01 &nbsp;|&nbsp; **Data:** daily OHLCV

---

## Abstract

We test whether a set of standard technical signals (momentum, RSI, volume, and price
z-score) carry *statistically significant* predictive power for the cross-section of
Indian equity returns. We evaluate each signal with the Information Coefficient (IC)
framework, adjust significance for autocorrelation using **Newey-West (HAC)** standard
errors, and control for data-mining across the full grid of tests with the
**Benjamini-Hochberg** false-discovery-rate procedure. We complement this with
**Fama-MacBeth** cross-sectional regressions for factor premia, a **turnover-adjusted**
long/short Sharpe ratio with a **bootstrap confidence interval**, five portfolio
construction schemes, and a **square-root market-impact** transaction-cost model.

**Headline result:** Of 20 (signal, horizon) hypotheses, only one survives FDR
correction — the **1-day price z-score**, a short-horizon *mean-reversion* effect
($t_{NW}=-3.24$, $p_{adj}=0.024$). Trend-following signals show no significant premium,
and a naive momentum long/short **loses money once realistic turnover costs are applied**.

## 1. Methodology

### 1.1 Information Coefficient
For a signal $s$ and horizon $h$, on each date $t$ we compute the cross-sectional
Spearman rank correlation between the signal and the subsequent $h$-day return:
$$ IC_t = \rho_{\text{Spearman}}\big(s_{i,t},\; r_{i,t\to t+h}\big). $$
The **Information Coefficient Information Ratio** summarizes signal quality:
$$ IC\text{-}IR = \frac{\overline{IC}}{\sigma_{IC}}. $$

### 1.2 Newey-West significance
Because overlapping forward windows induce autocorrelation, we test
$H_0:\overline{IC}=0$ using **HAC** standard errors with
$L=\lfloor 4(T/100)^{2/9}\rfloor$ lags (Newey & West, 1987).

### 1.3 Multiple-testing control
Testing many signals inflates false positives. We apply **Benjamini-Hochberg**
(1995) to control the false discovery rate at $\alpha=0.05$; a signal is “real” only
if its *adjusted* p-value clears the threshold.

### 1.4 Fama-MacBeth premia
Each period we regress forward returns on the standardized exposure and average the
slopes: $\hat{\lambda} = \frac{1}{T}\sum_t \hat{\lambda}_t$, tested with NW errors
(Fama & MacBeth, 1973).

### 1.5 Net-of-cost performance
We build a daily long/short book (top vs bottom tercile), charge
$\text{cost}_t = \text{turnover}_t \times c$ with $c=10$ bps, and bootstrap the
Sharpe ratio for a 95% confidence interval.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.research.statistical_validation import StatisticalValidator
from src.portfolio.optimizer import PortfolioOptimizer
from src.execution.market_impact import ExecutionAnalyzer

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
print('Environment ready.')

## 2. Signal Efficacy: IC, IC-IR, and Significance

We run the full validation suite and display the IC table. The `significant` column
reflects the Benjamini-Hochberg-adjusted decision.

In [ ]:
validator = StatisticalValidator()
report = validator.run()   # downloads (cached) universe and runs everything

ic_df = pd.DataFrame([{
    'signal': r.factor, 'horizon': r.horizon, 'mean_IC': r.mean_ic,
    'IC_IR': r.ic_ir, 'NW_t': r.nw_tstat, 'p_value': r.pvalue,
    'p_adj': r.pvalue_adj, 'significant': r.significant,
} for r in report.ic_results])
ic_df

In [ ]:
# Which hypotheses survive the false-discovery-rate correction?
survivors = ic_df[ic_df['significant']]
print(f'Hypotheses tested : {len(ic_df)}')
print(f'Significant (FDR) : {len(survivors)}')
survivors

In [ ]:
# Visualize IC-IR by signal and horizon. Bars beyond the dashed guides are the
# economically interesting ones; only FDR-significant cells are truly reliable.
pivot = ic_df.pivot(index='signal', columns='horizon', values='IC_IR')
ax = pivot.plot(kind='bar')
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('IC-IR')
ax.set_title('Signal quality (IC-IR) by horizon')
ax.legend(title='Horizon (days)')
plt.tight_layout(); plt.show()

**Reading the table.** Negative mean IC on the momentum/RSI signals indicates that,
over this sample, stocks that recently *rose* tended to slightly *underperform* — i.e.
short-horizon **mean reversion** dominated trend. Crucially, after Newey-West and
Benjamini-Hochberg, only the **1-day price z-score** clears the significance bar. This
is the disciplined-researcher conclusion: *most apparent edges are noise.*

## 3. Fama-MacBeth Factor Premia

The average per-period slope $\hat{\lambda}$ is the estimated premium per one standard
deviation of exposure. Starred rows are significant at 5% (raw NW).

In [ ]:
fm_df = pd.DataFrame([{
    'signal': r.factor, 'horizon': r.horizon, 'premium': r.mean_premium,
    'NW_t': r.nw_tstat, 'p_value': r.pvalue, 'n_periods': r.n_periods,
} for r in report.fm_results])
fm_df['sig_5pct'] = fm_df['p_value'] < 0.05
fm_df

## 4. Net-of-Cost Performance and Bootstrap Uncertainty

A signal is only tradeable if its edge survives transaction costs. We report gross vs
net Sharpe for the momentum long/short, with a 95% bootstrap confidence interval on
the *net* Sharpe.

In [ ]:
t = report.turnover_result
summary = pd.Series({
    'Gross Sharpe': t.gross_sharpe,
    'Net Sharpe': t.net_sharpe,
    'Net Sharpe CI low': t.sharpe_ci_low,
    'Net Sharpe CI high': t.sharpe_ci_high,
    'Annual gross return %': t.annual_gross_return * 100,
    'Annual net return %': t.annual_net_return * 100,
    'Annual turnover (x)': t.annual_turnover,
})
summary.to_frame(name=f'Long/Short {t.factor}')

In [ ]:
fig, ax = plt.subplots()
ax.bar(['Gross', 'Net'], [t.gross_sharpe, t.net_sharpe],
       color=['#4C72B0', '#C44E52'])
ax.errorbar(['Net'], [t.net_sharpe],
            yerr=[[t.net_sharpe - t.sharpe_ci_low], [t.sharpe_ci_high - t.net_sharpe]],
            fmt='none', ecolor='k', capsize=6)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Annualized Sharpe')
ax.set_title('Momentum L/S: cost destroys the (already weak) edge')
plt.tight_layout(); plt.show()
print(f'Turnover-cost drag: {(t.annual_gross_return - t.annual_net_return)*100:.1f}% / yr')

## 5. Portfolio Construction

Given a basket, we compare five allocation schemes. Note how **Risk Parity** and
**HRP** equalize *risk* rather than *capital*, and **Black-Litterman** (no views)
recovers the equilibrium prior.

In [ ]:
basket = ['RELIANCE','TCS','HDFCBANK','SBIN','BHARTIARTL',
          'SUNPHARMA','COALINDIA','NTPC','BAJFINANCE','ITC']
opt = PortfolioOptimizer().load(basket)

rows = []
weights_by_method = {}
for m in ['max_sharpe','min_variance','risk_parity','hrp','black_litterman']:
    res = opt.optimize(m)
    weights_by_method[m] = res.weights
    rows.append({'method': res.method, 'exp_return_%': res.expected_return*100,
                 'volatility_%': res.volatility*100, 'sharpe': res.sharpe,
                 'diversification': res.diversification_ratio})
pd.DataFrame(rows).set_index('method')

In [ ]:
wdf = pd.DataFrame(weights_by_method) * 100
ax = wdf.plot(kind='bar')
ax.set_ylabel('Weight (%)')
ax.set_title('Allocation by method')
ax.legend(title='Method', fontsize=8)
plt.tight_layout(); plt.show()

## 6. Transaction-Cost Model

The square-root law estimates price impact as
$$ \text{impact (bps)} = c\,\sigma\,\sqrt{Q/\text{ADV}}\times 10^4, $$
so cost grows with the **square root** of participation. We illustrate the cost curve
for RELIANCE across order sizes.

In [ ]:
ex = ExecutionAnalyzer()
order_values = [1e6, 5e6, 1e7, 5e7, 1e8, 5e8]
curve = [ex.estimate_impact('RELIANCE', order_value=v) for v in order_values]
cost_df = pd.DataFrame({
    'order_value_Rs': order_values,
    'participation_%': [e.participation*100 for e in curve],
    'impact_bps': [e.impact_bps for e in curve],
    'total_cost_bps': [e.total_cost_bps for e in curve],
})
cost_df

In [ ]:
ax = cost_df.plot(x='participation_%', y='impact_bps', marker='o', legend=False)
ax.set_xlabel('Participation (% of ADV)')
ax.set_ylabel('Market impact (bps)')
ax.set_title('Square-root market-impact curve — RELIANCE')
plt.tight_layout(); plt.show()

## 7. Conclusion

1. **Signal efficacy is rare.** Across 20 hypotheses, only the **1-day price z-score**
   survives Newey-West + Benjamini-Hochberg ($t_{NW}=-3.24$, $p_{adj}=0.024$), a
   short-horizon mean-reversion effect. Trend/momentum signals are not significant on
   this sample.
2. **Costs are decisive.** The momentum long/short is roughly flat gross but turns
   sharply negative net of a 10 bps cost at ~165x annual turnover — a textbook case of
   an “edge” that does not survive frictions.
3. **Construction matters.** Risk-based allocation (Risk Parity, HRP) produces more
   balanced risk contributions than capital-weighting or unstable mean-variance.
4. **Impact is non-linear.** Trading cost scales with $\sqrt{Q/\text{ADV}}$; sizing and
   scheduling (Almgren-Chriss) are first-order to realized performance.

**Takeaway.** The contribution here is not a new feature but *evidence*: a reproducible
pipeline that separates genuine signal from noise, prices frictions honestly, and sizes
positions by risk. Future work: extend to a broader survivorship-bias-free universe,
add point-in-time fundamentals for value/quality premia, and incorporate an
alternative-data signal (e.g., FinBERT on earnings-call transcripts).

---
*References: Fama & MacBeth (1973); Newey & West (1987); Benjamini & Hochberg (1995);
Grinold & Kahn (2000); Almgren & Chriss (2000); López de Prado (2016).*